## Install required dependencies

In [ ]:
!pip install ultralytics

In [ ]:
!pip install --force-reinstall torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install "pillow<10" --force-reinstall

In [ ]:
!pip install "numpy<2" --force-reinstall

## Prepare Dataset

In [1]:
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

In [2]:
# ===== CONFIG =====
RICO_PATH = "/kaggle/input/datasets/ameshmjayaweera/rico-mobile-uis/Rico"
OUT_PATH = "/kaggle/working/data"
VAL_FRAC = 0.2
SEED = 42
LINK_MODE = "copy"   # use "copy" in Kaggle

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla P100-PCIE-16GB


### Classes

In [4]:
CLASSES = [
    "Text","Icon","Image","TextButton","UpperTaskBar","PageIndicator",
    "CheckedTextView","EditText","BackgroundImage","Modal","Toolbar",
    "Drawer","Switch","Card",
]

DROPPED = {"Multi_Tab", "Bottom_Navigation", "Spinner", "Map"}
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

In [5]:
CLASS_TO_ID

{'Text': 0,
 'Icon': 1,
 'Image': 2,
 'TextButton': 3,
 'UpperTaskBar': 4,
 'PageIndicator': 5,
 'CheckedTextView': 6,
 'EditText': 7,
 'BackgroundImage': 8,
 'Modal': 9,
 'Toolbar': 10,
 'Drawer': 11,
 'Switch': 12,
 'Card': 13}

### Helpers

In [6]:
def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    width = int(size.findtext("width", "0"))
    height = int(size.findtext("height", "0"))

    objs = []
    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip()
        bnd = obj.find("bndbox")
        if not name or bnd is None:
            continue

        xmin = float(bnd.findtext("xmin", "0"))
        ymin = float(bnd.findtext("ymin", "0"))
        xmax = float(bnd.findtext("xmax", "0"))
        ymax = float(bnd.findtext("ymax", "0"))

        objs.append((name, xmin, ymin, xmax, ymax))

    return width, height, objs


def to_yolo_line(name, xmin, ymin, xmax, ymax, w, h):
    if name not in CLASS_TO_ID:
        return None

    xmin = max(0, min(w, xmin))
    ymin = max(0, min(h, ymin))
    xmax = max(0, min(w, xmax))
    ymax = max(0, min(h, ymax))

    if xmax <= xmin or ymax <= ymin:
        return None

    cx = (xmin + xmax) / 2 / w
    cy = (ymin + ymax) / 2 / h
    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h

    return f"{CLASS_TO_ID[name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"

### Shuffle Dataset

In [7]:
rico_root = Path(RICO_PATH)
ann_dir = rico_root / "Annotations"
img_dir = rico_root / "JPEGImages"
out_root = Path(OUT_PATH)

xml_files = sorted(ann_dir.glob("*.xml"))

pairs = []
for xml_path in xml_files:
    jpg_path = img_dir / f"{xml_path.stem}.jpg"
    if jpg_path.exists():
        pairs.append((xml_path, jpg_path))

# Shuffle + split
random.seed(SEED)
random.shuffle(pairs)

val_count = int(len(pairs) * VAL_FRAC)
val_pairs = pairs[:val_count]
train_pairs = pairs[val_count:]

splits = {"train": train_pairs, "val": val_pairs}

In [8]:
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    d = out_root / sub
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

In [9]:
per_split_counts = {"train": Counter(), "val": Counter()}

for split_name, split_pairs in splits.items():
    img_out = out_root / "images" / split_name
    lbl_out = out_root / "labels" / split_name

    for xml_path, jpg_path in split_pairs:
        w, h, objs = parse_voc_xml(xml_path)

        lines = []
        for name, xmin, ymin, xmax, ymax in objs:
            line = to_yolo_line(name, xmin, ymin, xmax, ymax, w, h)
            if line:
                lines.append(line)
                per_split_counts[split_name][name] += 1

        # write label
        (lbl_out / f"{xml_path.stem}.txt").write_text("\n".join(lines))

        # copy image
        shutil.copy2(jpg_path, img_out / jpg_path.name)

### Create data.yaml

In [10]:
yaml_path = out_root / "data.yaml"

yaml_content = [
    f"path: {out_root}",
    "train: images/train",
    "val: images/val",
    "names:",
]

for i, name in enumerate(CLASSES):
    yaml_content.append(f"  {i}: {name}")

yaml_path.write_text("\n".join(yaml_content))

print("Dataset ready at:", out_root)
print("YAML:", yaml_path)

Dataset ready at: /kaggle/working/data
YAML: /kaggle/working/data/data.yaml


## Taining

In [11]:
def train(
    data="data/data.yaml",
    weights="yolo11n.pt",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device="",
    project="runs",
    name="rico_yolo11n",
    seed=42,
    workers=4,
    resume=False,
):
    from pathlib import Path
    from ultralytics import YOLO

    data_path = Path(data).resolve()
    if not data_path.exists():
        raise ValueError(f"data file not found at {data_path}")

    model = YOLO(weights)

    train_kwargs = dict(
        data=str(data_path),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=patience,
        cos_lr=True,
        pretrained=True,
        seed=seed,
        workers=workers,
        project=project,
        name=name,
        resume=resume,
    )

    if device:
        train_kwargs["device"] = device

    results = model.train(**train_kwargs)

    save_dir = getattr(results, "save_dir", None) or Path(project) / name
    print(f"Best weights: {Path(save_dir) / 'weights' / 'best.pt'}")

In [12]:
train(
    data="/kaggle/working/data/data.yaml",
    epochs=75,
    batch=16
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.2.2+cu118 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rico_yolo11n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

In [ ]:
# !nvidia-smi

In [ ]:
# !pip install --force-reinstall torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# pip install "numpy<2"

## Evaluation

In [15]:
from pathlib import Path
import shutil

from ultralytics import YOLO

# ===== CONFIG =====
WEIGHTS = "/kaggle/working/runs/detect/runs/rico_yolo11n/weights/best.pt"
DATA = "/kaggle/working/data/data.yaml"   # or /kaggle/input/... if not copied
PROJECT = "runs"
NAME = "rico_yolo11n_eval"

IMG_SIZE = 640
BATCH = 16
CONF = 0.001
IOU = 0.6
DEVICE = "0"   # use "cpu" if no GPU

In [16]:
weights_path = Path(WEIGHTS)
data_path = Path(DATA)

assert weights_path.exists(), f"Weights not found: {weights_path}"
assert data_path.exists(), f"Data not found: {data_path}"

print("Weights:", weights_path)
print("Data:", data_path)

Weights: /kaggle/working/runs/detect/runs/rico_yolo11n/weights/best.pt
Data: /kaggle/working/data/data.yaml


In [17]:
model = YOLO(str(weights_path))
print("Model loaded")

Model loaded


In [18]:
metrics = model.val(
    data=str(data_path),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    conf=CONF,
    iou=IOU,
    plots=True,
    save_json=False,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    device=DEVICE
)

save_dir = Path(metrics.save_dir)
print("Evaluation complete")
print("Results saved at:", save_dir)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.2.2+cu118 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11n summary (fused): 101 layers, 2,584,882 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1657.6±646.3 MB/s, size: 101.3 KB)
val: Scanning /kaggle/working/data/labels/val.cache... 400 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 400/400 139.8Mit/s 0.0s
val: /kaggle/working/data/images/val/5131.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 5.0it/s 5.0s0.1s
                   all        400       4933      0.884      0.869      0.909      0.848
                  Text        398       2205      0.963      0.944      0.974      0.869
                  Icon        226        802      0.888      0.807      0.897      0.769
                 Image        310        555      0.806      0.714      0.835      0.783
            TextButton        220        3

In [19]:
if hasattr(metrics, "box"):
    b = metrics.box

    print("\nMetrics:")
    print(f"Precision: {float(b.mp):.4f}")
    print(f"Recall   : {float(b.mr):.4f}")
    print(f"mAP@0.5  : {float(b.map50):.4f}")
    print(f"mAP@0.5:0.95: {float(b.map):.4f}")


Metrics:
Precision: 0.8835
Recall   : 0.8687
mAP@0.5  : 0.9088
mAP@0.5:0.95: 0.8476


In [20]:
PLOT_FILES = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
]

print("Generated plots:\n")

for f in PLOT_FILES:
    p = save_dir / f
    print("✔️" if p.exists() else "❌", p)

Generated plots:

✔️ /kaggle/working/runs/detect/runs/rico_yolo11n_eval/confusion_matrix.png
✔️ /kaggle/working/runs/detect/runs/rico_yolo11n_eval/confusion_matrix_normalized.png


In [21]:
COPY_TO = "/kaggle/working/eval_plots"

dst = Path(COPY_TO)
dst.mkdir(parents=True, exist_ok=True)

copied = 0
for f in PLOT_FILES:
    src = save_dir / f
    if src.exists():
        shutil.copy2(src, dst / f)
        copied += 1

print(f"Copied {copied}/{len(PLOT_FILES)} plots to {dst}")

Copied 2/2 plots to /kaggle/working/eval_plots
